# Credit Card Fraud Detection — Standard Autoencoder

**PAAI Course Project**

---

## How This Works

A **Standard Autoencoder** trained only on legitimate transactions.
It learns to compress and reconstruct normal transaction patterns.
Fraudulent transactions don't fit the learned pattern — their reconstruction error is high.

```
Input → [Encoder] → Bottleneck → [Decoder] → Reconstructed
                                                    ↓
                         MSE(original, reconstructed) = fraud score
```

## 1. Setup & Imports

In [4]:
!pip install kaggle --quiet

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_recall_curve,
    confusion_matrix, classification_report
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Load Dataset from Kaggle

Get your `kaggle.json` from: kaggle.com → Profile → Settings → API → Create New Token

In [5]:
path = kagglehub.dataset_download("tjverry/credit-card-transactions")
df = pd.read_csv(path + "/transactions.csv")
print(f'Dataset shape: {df.shape}')
df.head()

Using Colab cache for faster access to the 'credit-card-transactions' dataset.
Dataset shape: (786363, 22)


,accountNumber,creditLimit,availableMoney,transactionDateTime,merchantName,transactionAmount,acqCountry,merchantCountryCode,posEntryMode,posConditionCode,...,accountOpenDate,dateOfLastAddressChange,cardCVV,enteredCVV,cardLast4Digits,transactionType,currentBalance,cardPresent,expirationDateKeyInMatch,isFraud
0,737265056,5000,5000.0,2016-08-13T14:27:32,Uber,98.55,US,US,2.0,1.0,...,3/14/2015,3/14/2015,414,414,1803,PURCHASE,0.0,0,0,0
1,737265056,5000,5000.0,2016-10-11T05:05:54,AMC #191138,74.51,US,US,9.0,1.0,...,3/14/2015,3/14/2015,486,486,767,PURCHASE,0.0,1,0,0
2,737265056,5000,5000.0,2016-11-08T09:18:39,Play Store,7.47,US,US,9.0,1.0,...,3/14/2015,3/14/2015,486,486,767,PURCHASE,0.0,0,0,0
3,737265056,5000,5000.0,2016-12-10T02:14:50,Play Store,7.47,US,US,9.0,1.0,...,3/14/2015,3/14/2015,486,486,767,PURCHASE,0.0,0,0,0
4,830329091,5000,5000.0,2016-03-24T21:04:46,Tim Hortons #947751,71.18,US,US,2.0,1.0,...,8/6/2015,8/6/2015,885,885,3143,PURCHASE,0.0,1,0,0


## 3. Feature Engineering

- Temporal features from datetime (hour, day of week)
- Account age and days since address change
- Ratio features: amount/limit, amount/balance
- Binary flags: CVV match, cross-border, card present, expiry match
- One-hot encoded categoricals

In [6]:
def engineer_features(df):
    df = df.copy()

    df['transactionDateTime']     = pd.to_datetime(df['transactionDateTime'])
    df['accountOpenDate']         = pd.to_datetime(df['accountOpenDate'])
    df['currentExpDate']          = pd.to_datetime(df['currentExpDate'])
    df['dateOfLastAddressChange'] = pd.to_datetime(df['dateOfLastAddressChange'])

    df['transaction_hour'] = df['transactionDateTime'].dt.hour
    df['transaction_dow']  = df['transactionDateTime'].dt.dayofweek

    df['account_age_days'] = (df['transactionDateTime'] - df['accountOpenDate']).dt.days

    df['days_since_address_change'] = (
        df['transactionDateTime'] - df['dateOfLastAddressChange']
    ).dt.days

    df['months_until_expiry'] = (
        (df['currentExpDate'].dt.year  - df['transactionDateTime'].dt.year) * 12 +
        (df['currentExpDate'].dt.month - df['transactionDateTime'].dt.month)
    )

    df['amount_to_limit_ratio']   = df['transactionAmount'] / (df['creditLimit'] + 1)
    df['amount_to_balance_ratio'] = df['transactionAmount'] / (df['currentBalance'].abs() + 1)

    df['cvv_match']    = (df['cardCVV'] == df['enteredCVV']).astype(int)
    df['cross_border'] = (df['acqCountryCode'] != df['merchantCountryCode']).astype(int)
    df['card_present'] = df['cardPresent'].astype(int)
    df['expiry_match'] = df['expirationDateKeyInMatch'].astype(int)

    return df

df = engineer_features(df)
print('Feature engineering done.')

/tmp/ipykernel_19486/821588442.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['currentExpDate']          = pd.to_datetime(df['currentExpDate'])


OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 23-Jun, at position 0

In [ ]:
top_mccs = df['merchantCategoryCode'].value_counts().nlargest(20).index
df['merchantCategoryCode'] = df['merchantCategoryCode'].where(
    df['merchantCategoryCode'].isin(top_mccs), other='OTHER'
)

numerical_cols = [
    'transactionAmount', 'creditLimit', 'availableMoney', 'currentBalance',
    'amount_to_limit_ratio', 'amount_to_balance_ratio',
    'account_age_days', 'days_since_address_change', 'months_until_expiry',
    'transaction_hour', 'transaction_dow'
]

binary_cols = ['cvv_match', 'cross_border', 'card_present', 'expiry_match']

categorical_cols = [
    'transactionType', 'posEntryMode', 'posConditionCode',
    'merchantCategoryCode', 'acqCountryCode', 'merchantCountryCode'
]

df_encoded = pd.get_dummies(df[categorical_cols], drop_first=True)

X = pd.concat([
    df[numerical_cols].reset_index(drop=True),
    df[binary_cols].reset_index(drop=True),
    df_encoded.reset_index(drop=True)
], axis=1).fillna(0)

y = df['isFraud'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Fraud: {y.sum()} / {len(y)} ({100*y.mean():.2f}%)')

## 4. Train / Validation / Test Split

**Critical rule:** Train ONLY on non-fraud transactions.
Fraud samples appear only in validation and test sets.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15/0.85, random_state=42, stratify=y_train_full
)

# Fit scaler ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Keep only normal transactions for training
X_train_normal = X_train_scaled[y_train == 0]

print(f'Training on {len(X_train_normal)} normal transactions')
print(f'Validation:  {len(X_val)} samples ({y_val.sum()} fraud)')
print(f'Test:        {len(X_test)} samples ({y_test.sum()} fraud)')

## 5. PyTorch Dataset & DataLoader

In [ ]:
def to_tensor(arr):
    return torch.FloatTensor(arr).to(device)

train_tensor = to_tensor(X_train_normal)
val_tensor   = to_tensor(X_val_scaled)
test_tensor  = to_tensor(X_test_scaled)

train_loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=256, shuffle=True
)

input_dim = X_train_normal.shape[1]
print(f'Input feature size: {input_dim}')

## 6. Autoencoder Architecture

Hourglass shape — compress to bottleneck, then reconstruct:

```
Input (N) → 128 → 64 → 32 [bottleneck] → 64 → 128 → Output (N)
```

- **BatchNorm** stabilizes training across features with different scales
- **Dropout 0.2** prevents the model from overfitting to normal transactions
- No noise — the input is passed through the network as-is

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),         nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),   nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 128),  nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Autoencoder(input_dim=input_dim).to(device)
print(model)

## 7. Training

MSE loss measures reconstruction quality.
Early stopping halts training when validation loss stops improving.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

EPOCHS, PATIENCE = 50, 5
best_val_loss, patience_counter, best_state = float('inf'), 0, None
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for (batch,) in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    model.eval()
    with torch.no_grad():
        val_normal = val_tensor[torch.tensor(y_val == 0)]
        val_loss   = criterion(model(val_normal), val_normal).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | Train: {avg_train:.5f} | Val: {val_loss:.5f}')

model.load_state_dict(best_state)
print(f'Best val loss: {best_val_loss:.5f}')

In [ ]:
sns.lineplot(x=range(len(train_losses)), y=train_losses, label='Train')
sns.lineplot(x=range(len(val_losses)),   y=val_losses,   label='Val')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('Training Curve')

## 8. Reconstruction Error as Fraud Score

High MSE = model cannot reconstruct this transaction well = likely fraud.

In [ ]:
model.eval()
with torch.no_grad():
    test_recon = model(test_tensor)
    errors = ((test_tensor - test_recon) ** 2).mean(dim=1).cpu().numpy()

results_df = pd.DataFrame({'error': errors, 'isFraud': y_test})

print('Mean error — Normal:', results_df[results_df.isFraud==0]['error'].mean())
print('Mean error — Fraud: ', results_df[results_df.isFraud==1]['error'].mean())

In [ ]:
sns.histplot(data=results_df, x='error', hue='isFraud',
             bins=80, kde=True, log_scale=(True, False))
plt.title('Reconstruction Error: Fraud vs Normal')
plt.xlabel('MSE (log scale)')

## 9. Threshold Selection

Find the threshold that maximises F1 on the validation set.

In [ ]:
model.eval()
with torch.no_grad():
    val_errors = ((val_tensor - model(val_tensor)) ** 2).mean(dim=1).cpu().numpy()

precisions, recalls, thresholds = precision_recall_curve(y_val, val_errors)
f1_scores  = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx   = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f'Best threshold:  {best_threshold:.5f}')
print(f'F1 (val):        {f1_scores[best_idx]:.4f}')
print(f'Precision (val): {precisions[best_idx]:.4f}')
print(f'Recall (val):    {recalls[best_idx]:.4f}')

sns.lineplot(x=recalls, y=precisions)
plt.scatter(recalls[best_idx], precisions[best_idx], color='red', zorder=5, label='Best F1')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Validation Set)')
plt.legend()

## 10. Evaluation on Test Set

In [ ]:
y_pred = (errors > best_threshold).astype(int)

print('── Test Set Results ─────────────────────')
print(f'AUROC: {roc_auc_score(y_test, errors):.4f}   (target > 0.90)')
print(f'AUPRC: {average_precision_score(y_test, errors):.4f}   (target > 0.40)')
print(f'F1:    {f1_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal', 'Pred Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'])
plt.title('Confusion Matrix — Test Set')

## 11. Save the Model

In [ ]:
import pickle

torch.save(model.state_dict(), 'fraud_autoencoder_weights.pt')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('threshold.txt', 'w') as f:
    f.write(str(best_threshold))

print('Saved: fraud_autoencoder_weights.pt, scaler.pkl, threshold.txt')